# ETL Run Notebook
This notebook reproduces the procedure from `src/main.py`. Run cells in order.

***NOTE***: To recreate the current airfiles data must be pulled from 2003 to at least 2023 [2003,>=2023] 

## Import Library Dependencies

In [ ]:
import os, sys
from pathlib import Path

# Add workspace root to Python path to enable imports
workspace_root = Path.cwd().parent
sys.path.insert(0, str(workspace_root))
print(f"Added to sys.path: {workspace_root}")

from src import WeatherDataEtl

## Define User Inputs
- **Master Control** workbook path.

In [ ]:
master_control_file = Path(r'C:\Users\jon.gendron\Projects\lspc\lspc-climate-processing-restructure\test\test_inputs\Master_Control.xlsx')

## Instantiate the System 
- Wire application components

In [ ]:
app = WeatherDataEtl()


## Configure Projects
- ingesting -> validating -> objectifying user-input control workbooks
- create storage repository

### Ingest -> Validate -> Store (as Objects)
1. **Ingests** user-input data from master control workbook and project control workbooks
2. **Validates** data against predefined schemas and performs cross-validate checks for dependencies.
3. **Stores** data is a standard `ProjectControl` object that can be provided as input to run any module.

In [ ]:
projects = app.builder.build(master_control_file=master_control_file)
print(f'Found {len(projects)} project(s)')

### Builds storage repository for projects
- Based on Storage Sheet in project respoitory
- Does not overwrite existing respositories.

In [ ]:
for project in projects:
    app.builder.build_storage_respository(project)

## Run ETL Fetching for all sources

In [ ]:
for project in projects:
    app.run.fetch_weather_data(project)

## Run ETL Staging for all sources (except gage data) and QC for Raw gage data 

In [ ]:
for project in projects:
    app.run.stage_weather_data(project)
    app.run.qc_gage_data(project)

**NOTE: Natural break for manual qaqc**
Inspect staged data and perform manual QA/QC before continuing.

## Run ETL Staging for gage data after manual QAQC and generating QC'ed Intermediate files

In [ ]:
for project in projects:
    app.run.qc_remake_timeseries_automation(project)
    app.run.stage.gage(project)

## Run ETL Loading / Writing LSPC files

In [ ]:
for project in projects:
    app.run.write_lspc_files(project)